# Notebook 04 — GReaT (LLM) Synthetic Generation
Generate **240,000 synthetic sessions** using GReaT (Generation of Realistic Tabular data).

GReaT serialises each row as a natural-language sentence and fine-tunes GPT-2
on those sentences, then samples autoregressively. This produces strong
**cross-feature coherence** (REVERSE_SHELL co-occurs with high entropy + port 4444)
that diffusion models sometimes miss.

**Run on:** College system (RTX 4500 Ada)  
**Expected time:** ~6 hours fine-tuning + ~1 hour sampling

In [ ]:
import sys, logging, warnings, subprocess, hashlib
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger()

ROOT      = Path("..").resolve()
DATA_PROC = ROOT / "data" / "processed"
DATA_SYNTH= ROOT / "data" / "synthetic"
sys.path.insert(0, str(ROOT))

X_real = np.load(DATA_PROC / "X_real.npy")
y_real = np.load(DATA_PROC / "y_real.npy")
print(f"Real data: X={X_real.shape}  y={y_real.shape}")

## 4.1 Install GReaT

In [ ]:
try:
    import be_great
    print("GReaT already installed")
except ImportError:
    print("Installing be_great...")
    subprocess.run(["pip", "install", "be-great", "-q"], check=True)
    print("Installed")

## 4.2 Prepare Training DataFrame for GReaT

In [ ]:
from configs.schema import FEAT_NAMES, IDX_TO_LABEL

# GReaT needs a clean DataFrame — round floats for readability in sentences
df_great_input = pd.DataFrame(X_real, columns=FEAT_NAMES)
df_great_input["micro_state"] = [IDX_TO_LABEL[y] for y in y_real]

# Subsample for fine-tuning speed (50k representative rows is sufficient)
# GReaT fine-tuning scales with corpus size, not target generation size
GREAT_TRAIN_SAMPLE = 50_000
if len(df_great_input) > GREAT_TRAIN_SAMPLE:
    df_great_train = df_great_input.groupby("micro_state", group_keys=False).apply(
        lambda x: x.sample(min(len(x), GREAT_TRAIN_SAMPLE // 45 + 1), random_state=42)
    )
else:
    df_great_train = df_great_input

print(f"GReaT fine-tuning corpus: {len(df_great_train):,} rows")
print(f"Classes represented: {df_great_train['micro_state'].nunique()}/45")

## 4.3 Fine-tune GReaT on GPT-2

In [ ]:
from be_great import GReaT

GREAT_CKPT = DATA_SYNTH / "great_checkpoints"
GREAT_CKPT.mkdir(parents=True, exist_ok=True)

model = GReaT(
    llm="gpt2",                  # base GPT-2 (124M params) — fits in 24GB easily
    batch_size=32,                # college system; use 8 on laptop with fp16
    epochs=50,
    save_steps=2000,
    logging_steps=200,
    experiment_dir=str(GREAT_CKPT),
)

print("Starting GReaT fine-tuning...")
print(f"  Corpus size: {len(df_great_train):,} rows")
print(f"  Epochs: 50")
print(f"  Expected time (RTX 4500 Ada): ~5-6 hours")
print()
print("⚡ Uncomment the line below to start fine-tuning:")
print("   model.fit(df_great_train)")
print("   model.save(str(GREAT_CKPT / 'final_model'))")

# Uncomment to actually train:
# model.fit(df_great_train)
# model.save(str(GREAT_CKPT / "final_model"))
# print("Fine-tuning complete, model saved")

## 4.4 Sample 240,000 Synthetic Sessions

In [ ]:
N_SAMPLE_GREAT = 240_000

print(f"Sampling {N_SAMPLE_GREAT:,} sessions from fine-tuned GReaT model...")
print("⚡ Uncomment below to sample after fine-tuning completes:")
print()
print('   model = GReaT.load_from_dir(str(GREAT_CKPT / "final_model"))')
print('   df_great_raw = model.sample(n_samples=N_SAMPLE_GREAT, k=100, max_length=512)')
print('   df_great_raw.to_csv(DATA_SYNTH / "great_raw.csv", index=False)')

# Uncomment to run:
# model = GReaT.load_from_dir(str(GREAT_CKPT / "final_model"))
# df_great_raw = model.sample(n_samples=N_SAMPLE_GREAT, k=100, max_length=512)
# df_great_raw.to_csv(DATA_SYNTH / "great_raw.csv", index=False)

great_csv = DATA_SYNTH / "great_raw.csv"
if great_csv.exists():
    df_great_raw = pd.read_csv(great_csv)
    print(f"\nLoaded GReaT samples: {len(df_great_raw):,} rows")
else:
    print(f"\n'great_raw.csv' not found yet — run sampling above first.")

## 4.5 Post-Process GReaT Output
GReaT occasionally produces malformed rows (GPT-2 can break the structured format). Clean and validate before use.

In [ ]:
if 'df_great_raw' not in dir():
    print("⚠  Load great_raw.csv first (run section 4.4)")
else:
    from configs.schema import LABEL_TO_IDX, IDX_TO_LABEL, FEAT_NAMES
    from src.generators.kill_chain_simulator import (
        generate_session_sequence, is_valid_sequence, is_phase_monotone,
        filter_invalid_sequences
    )
    from src.generators.epss_drift import inject_multi_cve_drift
    import random

    # ── 1. Drop malformed rows (GPT-2 can emit broken numeric strings) ──
    n_before = len(df_great_raw)
    for col in FEAT_NAMES:
        df_great_raw[col] = pd.to_numeric(df_great_raw[col], errors="coerce")
    df_great_raw = df_great_raw.dropna(subset=FEAT_NAMES + ["micro_state"])
    n_after = len(df_great_raw)
    print(f"Malformed row cleanup: {n_before:,} → {n_after:,} "
          f"({n_before-n_after:,} dropped, {(n_before-n_after)/n_before*100:.1f}%)")

    # ── 2. Validate micro_state labels against schema ──
    valid_labels = set(LABEL_TO_IDX.keys())
    df_great_raw = df_great_raw[df_great_raw["micro_state"].isin(valid_labels)]
    print(f"After label validation: {len(df_great_raw):,} rows")

    df_great = df_great_raw.copy()
    df_great["micro_state_idx"] = df_great["micro_state"].map(LABEL_TO_IDX)

    # ── 3. Generate kill-chain-valid sequences for each session ──
    # GReaT generates single rows; we attach a plausible DAG-valid sequence
    # ending at the generated label for kill-chain coherence
    rng = random.Random(123)
    sequences = []
    for label in df_great["micro_state"]:
        seq = generate_session_sequence(rng, force_start=label, min_len=1, max_len=6)
        sequences.append(",".join(seq))
    df_great["micro_state_sequence"] = sequences

    # ── 4. Filter any accidentally invalid sequences ──
    df_great = filter_invalid_sequences(df_great, seq_col="micro_state_sequence")

    # ── 5. Add metadata ──
    df_great["session_id"] = [
        "gr_" + hashlib.md5(str(i).encode()).hexdigest()[:8]
        for i in range(len(df_great))
    ]
    df_great["source"] = "great_synthetic"
    df_great["session_day"] = np.random.randint(0, 90, size=len(df_great))
    df_great["associated_cve"] = ""

    # ── 6. EPSS drift for high-risk synthetic sessions ──
    cve_schedule = [
        {"cve_id": "CVE-2023-44487", "kev_day": 10},
        {"cve_id": "CVE-2024-3400",  "kev_day": 45},
        {"cve_id": "CVE-2023-20198", "kev_day": 25},
    ]
    high_risk_mask = df_great["micro_state_idx"].isin(range(23, 45))
    for i, sch in enumerate(cve_schedule):
        chunk_mask = high_risk_mask & (df_great.index % len(cve_schedule) == i)
        df_great.loc[chunk_mask, "associated_cve"] = sch["cve_id"]
    df_great = inject_multi_cve_drift(df_great, cve_schedule, seed=99)

    print(f"\nFinal GReaT dataset: {len(df_great):,} sessions")
    print(f"Classes covered: {df_great['micro_state'].nunique()}/45")

## 4.6 Extract Feature Matrix + Save

In [ ]:
if 'df_great' in dir():
    X_great = df_great[FEAT_NAMES].values.astype(np.float32)
    y_great = df_great["micro_state_idx"].values.astype(np.int64)
    X_great = np.nan_to_num(X_great, nan=0.0, posinf=1e4, neginf=-1e4)

    print(f"GReaT feature matrix: {X_great.shape}")

    np.save(DATA_SYNTH / "X_great.npy", X_great)
    np.save(DATA_SYNTH / "y_great.npy", y_great)
    df_great[["session_id","micro_state","source","session_day",
              "associated_cve","micro_state_sequence"]].to_parquet(
        DATA_SYNTH / "great_meta.parquet", index=False)
    print(f"Saved X_great.npy {X_great.shape}  y_great.npy {y_great.shape}")

    # ── Quality check ──
    from src.validators.quality_checks import adversarial_auc
    n = min(len(X_real), len(X_great), 10_000)
    adv = adversarial_auc(X_real[:n], X_great[:n], n_sample=n)
    print(f"\nGReaT adversarial AUC: {adv['adversarial_auc']} → {adv['verdict']}")
else:
    print("⚠  Run section 4.5 first")